# 🏆 7회차 · 미니 데이콘 챌린지
## AI를 활용한 바이브 코딩 대회

오늘의 미션: **AI에게 코드를 짜달라고 시켜서, 두 가지 ML 챌린지의 점수를 최대한 끌어올리기**

### 두 챌린지
- **🎬 회귀 (Regression)**: K-드라마 평균 시청률 예측 — 평가 지표 **RMSE (낮을수록 좋음)**
- **🎯 분류 (Classification)**: 영화 흥행 등급 (망함/소박/중박/대박) 분류 — 평가 지표 **Accuracy (높을수록 좋음)**

### 규칙
- **제출 횟수 무제한** 🔥 — 마음껏 시도하세요
- 1등은 각 챌린지 별로 시상
- 개인전. 코드는 자기가 짜되, **AI의 도움**(Colab Gemini, ChatGPT, Claude 등)은 마음껏 활용
- 두 챌린지 다 도전해도 되고, 하나에 집중해도 됨

### 바이브 코딩이란?
> "이 데이터로 RandomForest 분류기 만들어서 정확도 최대화하는 코드 짜줘"

처럼 AI에게 자연어로 코드를 부탁하는 방식. 오늘은 이걸로 ML 대회를 풉니다.

---
# Part 0. 환경 설정

먼저 라이브러리 설치 + 서버 연결 정보 입력.

In [ ]:
# 패키지 설치 (1분 정도)
!pip install -q scikit-learn xgboost lightgbm pandas numpy

In [ ]:
# ===== 여기 두 변수를 본인 값으로 바꾸세요 =====

# 강사가 안내한 cloudflared URL 붙여넣기 (마지막 / 없이)
SERVER_URL = "https://여기에-강사가-알려준-URL.trycloudflare.com"

# 본인 닉네임 (학번 권장, 한글/영문 OK, 20자 이내)
NICKNAME = "원하는 닉네임 입력 ㄱㄱ"

# ===== 검증 =====
SERVER_URL = SERVER_URL.strip().rstrip("/")
NICKNAME = NICKNAME.strip()

assert SERVER_URL.startswith("https://"), \
    "❌ SERVER_URL이 https://로 시작해야 합니다"
assert "여기에" not in SERVER_URL, \
    "❌ SERVER_URL을 강사가 알려준 실제 URL로 바꾸세요"
assert NICKNAME and "여기에" not in NICKNAME and "닉네임" not in NICKNAME, \
    "❌ NICKNAME을 본인 학번/이름으로 바꾸세요 (placeholder를 그대로 두면 안 됨)"

print(f"✅ Server: {SERVER_URL}")
print(f"✅ Nick  : {NICKNAME}")

In [ ]:
# 서버 연결 테스트
import requests
r = requests.get(f"{SERVER_URL}/health", timeout=10)
print(r.json())

---
# Part 1. 헬퍼 함수 — 데이터 다운로드와 제출

이 셀들은 그대로 실행만 하면 됩니다.

In [ ]:
import pandas as pd
import numpy as np
import requests
from IPython.display import display, Markdown

# 일부 환경에서 Cloudflare 봇 차단 회피용 user-agent 위장
SESSION = requests.Session()
SESSION.headers["User-Agent"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"


def download_data(task: str):
    """train.csv와 test.csv를 받아 pandas DataFrame 두 개 반환"""
    from io import StringIO

    def fetch_csv(path):
        r = SESSION.get(f"{SERVER_URL}{path}", timeout=30)
        r.raise_for_status()
        return pd.read_csv(StringIO(r.text))

    tr = fetch_csv(f"/data/{task}/train.csv")
    te = fetch_csv(f"/data/{task}/test.csv")
    print(f"[{task}] train {tr.shape} / test {te.shape}")
    return tr, te


def _format_left(left):
    """남은 제출 횟수 표시 — None이면 무제한"""
    return "♾️ 무제한" if left is None else f"{left}회 남음"


def submit(task: str, predictions, nickname: str = None):
    """예측값을 서버에 제출하고 점수와 등수를 받아옴.

    Args:
        task: "regression" 또는 "classification"
        predictions: test.csv의 id 순서대로 예측값 리스트 또는 numpy array
        nickname: 미지정시 전역 NICKNAME 사용
    """
    nick = nickname or NICKNAME
    preds = list(np.asarray(predictions).tolist())

    r = SESSION.post(
        f"{SERVER_URL}/api/submit",
        json={"task": task, "nickname": nick, "predictions": preds},
        timeout=30,
    )

    if r.status_code != 200:
        display(Markdown(f"### ❌ 제출 실패 ({r.status_code})\n\n```\n{r.json()}\n```"))
        return None

    res = r.json()
    medal = "🥇" if res["rank"] == 1 else "🥈" if res["rank"] == 2 else "🥉" if res["rank"] == 3 else f"#{res['rank']}"
    display(Markdown(
        f"### ✅ 제출 성공!\n"
        f"- **이번 점수**: `{res['score_this']:.4f}` ({res['metric']})\n"
        f"- **본인 최고**: `{res['score_best']:.4f}`\n"
        f"- **현재 등수**: {medal}\n"
        f"- **누적 제출**: {res['submissions_used']}회 ({_format_left(res.get('submissions_left'))})"
    ))
    return res


def check_leaderboard(task: str, top=10):
    """현재 리더보드 상위 N명 출력"""
    rows = SESSION.get(f"{SERVER_URL}/api/leaderboard/{task}").json()
    if not rows:
        display(Markdown("아직 제출이 없습니다."))
        return
    table = ["| 순위 | 닉네임 | 점수 | 제출수 |", "|---|---|---|---|"]
    for i, r in enumerate(rows[:top], 1):
        medal = "🥇" if i == 1 else "🥈" if i == 2 else "🥉" if i == 3 else f"{i}"
        mark = " 👈 (나)" if r["nickname"] == NICKNAME else ""
        table.append(f"| {medal} | {r['nickname']}{mark} | {r['best_score']:.4f} | {r['submissions']} |")
    display(Markdown("\n".join(table)))


def check_quota(task: str):
    """본인의 현재 제출 횟수 확인"""
    r = SESSION.get(f"{SERVER_URL}/api/quota/{task}/{NICKNAME}").json()
    display(Markdown(f"**[{task}]** 사용: {r['used']}회 · 남음: **{_format_left(r['left'])}**"))
    return r

print("✅ 헬퍼 함수 준비 완료")

---
# Part 2. 🎬 회귀 챌린지 — K-드라마 시청률 예측

## 데이터 설명
- **목표**: `avg_rating` (평균 시청률 %) 예측
- **평가**: RMSE (Root Mean Squared Error) — **낮을수록 좋음**
- **피처**:
  - `genre`: 장르 (로맨스/사극/스릴러/코미디/의학/법정/판타지)
  - `lead_actor_tier`: 주연 배우 등급 (1=A급, 5=신인급)
  - `writer_experience`: 작가 경력 (년)
  - `channel`: 채널 (지상파/케이블/OTT)
  - `episodes`: 회차 수
  - `budget_billion_won`: 제작비 (억원)
  - `release_quarter`: 분기 (1-4)
  - `timeslot`: 시간대 (평일심야/평일저녁/주말저녁/주말심야)
  - `is_remake`: 리메이크 여부 (0/1)
  - `lead_actor_age`: 주연 나이

In [ ]:
# 데이터 다운로드
tr_reg, te_reg = download_data("regression")
tr_reg.head()

In [ ]:
# 분포 살펴보기
display(Markdown(f"""
- 학습 데이터: {len(tr_reg)}행
- 테스트 데이터: {len(te_reg)}행
- 타겟 평균: {tr_reg['avg_rating'].mean():.2f}%
- 타겟 표준편차: {tr_reg['avg_rating'].std():.2f}%
- 타겟 범위: {tr_reg['avg_rating'].min():.2f} ~ {tr_reg['avg_rating'].max():.2f}
"""))
tr_reg.describe(include="all")

## 📌 베이스라인 — 평균만 예측해서 첫 제출

일단 가장 단순한 모델로 한 번 제출해보고 감을 잡아보세요.

In [ ]:
# 베이스라인 0: 단순히 평균값을 전부 예측
predictions = np.full(len(te_reg), tr_reg['avg_rating'].mean())

# 제출 — 제출 무제한이니 마음껏 시도하세요
submit("regression", predictions)

## 🚀 여기서부터 바이브 코딩

아래 빈 셀들에서 **Gemini / ChatGPT / Claude** 등에게 코드를 부탁해보세요.

### 추천 프롬프트 예시
```
tr_reg와 te_reg DataFrame이 있어.
컬럼은 ['genre', 'lead_actor_tier', 'writer_experience', 'channel',
'episodes', 'budget_billion_won', 'release_quarter', 'timeslot',
'is_remake', 'lead_actor_age', 'avg_rating'] 이고 마지막 게 타겟이야.

RandomForestRegressor로 RMSE를 최소화하는 코드를 짜줘.
범주형 변수는 적절히 인코딩해야 해.
test 예측 결과를 'predictions'라는 numpy array에 담아줘.
```

### 시도해볼 만한 것들
- 모델: RandomForest, XGBoost, LightGBM, CatBoost
- 범주형 인코딩: One-hot, Label, Target encoding
- 피처 엔지니어링: 상호작용 항 (예: genre × channel)
- 하이퍼파라미터 튜닝: GridSearch, Optuna
- 앙상블: 여러 모델 예측의 평균

### ⚠️ 제출 전 체크리스트
- [ ] 예측값 개수가 `len(te_reg)` (2000)과 일치하는가?
- [ ] `te_reg`의 `id` 순서대로 예측했는가?
- [ ] 음수나 비현실적 값(>50%)이 없는가?

In [ ]:
# 여기서부터 본인 코드 — Gemini에 부탁하든 직접 짜든 자유
# 예시: from sklearn.ensemble import RandomForestRegressor


In [ ]:
# 빈 셀 - 자유롭게 실험


In [ ]:
# 빈 셀 - 자유롭게 실험


In [ ]:
# 제출
# submit("regression", my_predictions)

In [ ]:
# 회귀 리더보드 확인
check_leaderboard("regression")

---
# Part 3. 🎯 분류 챌린지 — 영화 흥행 등급

## 데이터 설명
- **목표**: `box_office_grade` (0=망함, 1=소박, 2=중박, 3=대박) 예측
- **평가**: Accuracy (정확도) — **높을수록 좋음**
- **피처**:
  - `genre`: 장르 (액션/로맨스/스릴러/코미디/SF/드라마/호러/애니메이션)
  - `director_tier`: 감독 등급 (1=A급, 5=신인급)
  - `lead_actor_tier`: 주연 배우 등급 (1-5)
  - `production_budget`: 제작비 (억원)
  - `runtime_min`: 상영 시간 (분)
  - `rating_age`: 관람등급 (전체관람가/12세/15세/청불)
  - `release_month`: 개봉월 (1-12)
  - `is_sequel`: 속편 여부 (0/1)
  - `is_imported`: 수입 여부 (0/1)
  - `screen_count`: 개봉 스크린 수

In [ ]:
# 데이터 다운로드
tr_clf, te_clf = download_data("classification")
tr_clf.head()

In [ ]:
# 클래스 분포 확인
display(Markdown("**클래스 분포 (학습 데이터):**"))
label_names = {0: '망함', 1: '소박', 2: '중박', 3: '대박'}
counts = tr_clf['box_office_grade'].value_counts().sort_index()
for k, v in counts.items():
    pct = v / len(tr_clf) * 100
    print(f"  {k} ({label_names[k]}): {v}개 ({pct:.1f}%)")
print(f"  ※ 불균형 데이터입니다 — 0:1:2:3 ≈ 40:30:20:10")

## 📌 베이스라인 — 최빈 클래스 예측

In [ ]:
# 베이스라인 0: 가장 많이 나오는 클래스(0=망함)만 예측
predictions = np.zeros(len(te_clf), dtype=int)

submit("classification", predictions)  # 약 40% 정확도 나옴

## 🚀 여기서부터 바이브 코딩

### 추천 프롬프트 예시
```
tr_clf와 te_clf DataFrame이 있어.
다중 클래스 분류(0,1,2,3)인데 클래스가 불균형이야 (40:30:20:10).
XGBoost로 정확도 최대화하는 코드를 짜줘.
class_weight나 sample_weight 같은 거 활용해도 돼.
test 예측을 'my_predictions'라는 정수 numpy array로 만들어줘.
```

### 시도해볼 것
- 모델: RandomForest, XGBoost, LightGBM, GradientBoosting
- 불균형 처리: `class_weight='balanced'`, SMOTE
- 임계값 조정 (예측 확률 → 클래스)
- 피처 엔지니어링: 비율, 곱셈 등

### ⚠️ 제출 전 체크리스트
- [ ] 예측값이 정수 0/1/2/3 (확률 아님)인가?
- [ ] 개수가 `len(te_clf)` (2000)과 일치하는가?

In [ ]:
# 여기서부터 본인 코드


In [ ]:
# 빈 셀


In [ ]:
# 빈 셀


In [ ]:
# 제출
# submit("classification", my_predictions)

In [ ]:
# 분류 리더보드 확인
check_leaderboard("classification")

In [ ]:
check_quota("regression")
check_quota("classification")
display(Markdown("---"))
display(Markdown("### 🎬 회귀 리더보드"))
check_leaderboard("regression", top=20)
display(Markdown("### 🎯 분류 리더보드"))
check_leaderboard("classification", top=20)

---
# Part 4. 🎊 마무리

## 본인 최종 점수 확인

## 오늘 배운 것
- **바이브 코딩이 진짜 가능하다** — AI에게 코드를 시키는 시대
- **데이터를 이해하는 게 출발점** — 모델 고르기 전에 분포부터 봐야 함
- **단순한 모델이 의외로 강력** — RandomForest, XGBoost 같은 트리 기반은 거의 모든 tabular 문제에서 강함
- **앙상블이 마지막 보너스** — 서로 다른 모델의 예측을 평균만 내도 점수가 오르는 경우가 많음

## 더 알아볼 키워드
- **Kaggle / 데이콘 / 캐글 한국** — 실제 글로벌 ML 대회
- **AutoML** (H2O, AutoGluon, FLAML) — AI가 모델 선택까지 자동화
- **Feature Engineering** — Kaggle 마스터들의 진짜 무기
- **Stacking** — 모델 예측을 다시 메타 모델의 입력으로 쓰는 고급 앙상블

수고하셨습니다 🏆